In [4]:
import numpy as np
import pandas as pd

import time
from tqdm import tqdm
from tvDatafeed import TvDatafeed, Interval

def get_data_from_tradingview(
    tickers: list[str],
    interval: Interval, # tvDatafeed.Interval 객체 형태
    exchange: list[str],
    n_bars: int, # 충분히 큰 수. Tradingview는 기간이 아닌 candle bar의 개수로 호출 가능
    column: str | None = None,
    verbose: bool = True, # 종목을 얼마만큼 호출 성공했는지 progress bar로 표현
    num_trials: int = 5, # 호출 실패 시 재시도 횟수
    multi_level_index: bool = True, # yfinance에서의 attribute와 동일한 기능
    tz_cleansing : bool = False,
    session_duration_map: dict[str, tuple[int, int]] | None = None,
) -> pd.DataFrame:
    """
    TradingView(tvDatafeed)로부터 가격데이터 import

    Params
    - column:
        * str  : 해당 컬럼만 반환 (columns=tickers)
        * None : OHLCV 전체 반환
    - multi_level_index:
        * True  : columns=MultiIndex (ticker, field)  -> ("AAPL","close") 형태
        * False : columns=MultiIndex (field, ticker)  -> data["close"] 로 모든 종목 접근 가능

    - tz_cleansing:
        * True  : 인덱스를 날짜 단위로 정규화
        * False : 원본 인덱스 유지

    - session_duration_map:
        * None : 인덱스 변경 없음
        * dict : 거래소별 open -> close 시간 차이
            예:
            {
                "NASDAQ": (6, 30),
                "NYSE": (6, 30),
                "KRX": (6, 30),
                "TSE": (6, 0),
            }
        지정된 거래소는 기존 인덱스에 해당 시간만큼 더함.

    Notes
    - column이 str이면 multi_level_index 설정은 의미가 거의 없고, 그냥 (columns=tickers)로 반환.
    """

    def _shift_index_by_session_duration(
        idx: pd.Index,
        exch: str,
        session_duration_map: dict[str, tuple[int, int]] | None,
    ) -> pd.DatetimeIndex:
        idx = pd.to_datetime(idx)

        if getattr(idx, "tz", None) is not None:
            idx = idx.tz_localize(None)

        if session_duration_map is None:
            return pd.DatetimeIndex(idx)

        if exch not in session_duration_map:
            return pd.DatetimeIndex(idx)

        hours, minutes = session_duration_map[exch]
        delta = pd.Timedelta(hours=hours, minutes=minutes)

        return pd.DatetimeIndex(idx + delta)

    tv = TvDatafeed()
    iterator = tqdm(list(zip(tickers, exchange)), disable=not verbose)

    ohlcv_cols = ["open", "high", "low", "close", "volume"]
    frames: list[pd.DataFrame] = []
    series_list: list[pd.Series] = []

    for ticker, exch in iterator:
        got = False

        for attempt in range(num_trials):
            try:
                temp = tv.get_hist(
                    symbol=ticker,
                    exchange=exch,
                    interval=interval,
                    n_bars=n_bars,
                )

                if temp is None or temp.empty:
                    raise ValueError(f"Empty data returned for {ticker} ({exch}).")

                temp.columns = [c.lower() for c in temp.columns]
                temp.index = pd.to_datetime(temp.index)

                # 거래소별 open -> close shift
                if session_duration_map is not None:
                    temp.index = _shift_index_by_session_duration(
                        idx=temp.index,
                        exch=exch,
                        session_duration_map=session_duration_map,
                    )

                elif tz_cleansing:
                    temp.index = pd.to_datetime(temp.index.strftime("%Y-%m-%d"))

                else:
                    if getattr(temp.index, "tz", None) is not None:
                        temp.index = temp.index.tz_localize(None)

                if column is None:
                    use_cols = [c for c in ohlcv_cols if c in temp.columns]
                    temp2 = temp[use_cols].copy()
                    temp2.columns = pd.MultiIndex.from_product(
                        [[ticker], temp2.columns.tolist()],
                        names=["ticker", "field"],
                    )
                    frames.append(temp2)
                else:
                    col = column.lower()
                    if col not in temp.columns:
                        raise KeyError(
                            f"Column '{column}' not in data columns for {ticker}: {list(temp.columns)}"
                        )
                    series_list.append(temp[col].rename(ticker))

                got = True
                break

            except Exception as e:
                if attempt < num_trials - 1:
                    time.sleep(1)
                else:
                    print(f"[FAIL] {ticker} ({exch}) after {num_trials} trials: {e}")

        if not got:
            continue

    if column is not None:
        if not series_list:
            return pd.DataFrame()
        return pd.concat(series_list, axis=1).sort_index()

    if not frames:
        return pd.DataFrame()

    out = pd.concat(frames, axis=1).sort_index()

    if multi_level_index:
        return out.sort_index(axis=1)

    out.columns = out.columns.swaplevel(0, 1)
    return out.sort_index(axis=1)

In [5]:
asset_data = {
    'KOSPI' : 'KRX', # KOSPI Composite
    'KOSPI200' : 'KRX', # KOSPI200
    'KOSDAQ' : 'KRX', # KOSDAQ
    'KOSDAQ150' : 'KRX',
    'SPX' : 'TVC', # S&P500
    'NDQ' : 'TVC', # NASDAQ
    'DJI' : 'TVC', # Dow Jones
    'SX5E': 'TVC', # STOXX50
    'NI225' : 'TVC', # NIKKEI225
    '000001' : 'SSE' # SSEC
}

In [6]:
data = get_data_from_tradingview(
    tickers = list(asset_data.keys()),
    interval = Interval.in_daily,
    exchange = list(asset_data.values()),
    n_bars = 13000,
    column = 'close',
    verbose = True,
    num_trials = 5,
    multi_level_index = False,
    tz_cleansing = True
)

100%|██████████| 10/10 [00:40<00:00,  4.05s/it]
C:\Users\krm\AppData\Local\Temp\ipykernel_22656\1355288727.py:143: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  return pd.concat(series_list, axis=1).sort_index()


In [9]:
data

,KOSPI,KOSPI200,KOSDAQ,KOSDAQ150,SPX,NDQ,DJI,SX5E,NI225,000001
datetime,,,,,,,,,,
1973-09-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4677.64,NaN
1973-09-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4696.49,NaN
1973-09-11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4587.40,NaN
1973-09-12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4646.37,NaN
1973-09-13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4620.91,NaN
...,...,...,...,...,...,...,...,...,...,...
2026-05-18,7516.04,1171.30,1111.09,1865.26,7403.04,28994.3682,49690.9586,5849.01,60815.73,4131.5276
2026-05-19,7271.66,1132.42,1084.36,1823.80,7353.62,28818.8440,49368.9464,5851.17,60550.37,4169.5378
2026-05-20,7208.95,1125.51,1056.07,1773.46,7432.96,29297.6982,50013.9909,5976.08,59804.19,4162.1845


In [10]:
data.columns = [
    'KOSPI','KOSPI200','KOSDAQ','KOSDAQ150','SNP500','NASDAQ','DOW_JONES','EURO_STOXX_50','NIKKEI_225','SHANGHAI_COMP'
]

In [21]:
data.to_excel('testing.xlsx')

In [17]:
data_long = data.loc['2020':].unstack().reset_index()

In [18]:
data_long.columns = ['INDC_CD','BASE_YMD','INDX_VAL']

In [20]:
data_long.dropna()

,INDC_CD,BASE_YMD,INDX_VAL
0,KOSPI,2020-01-02,2175.1699
1,KOSPI,2020-01-03,2176.4600
2,KOSPI,2020-01-06,2155.0701
3,KOSPI,2020-01-07,2175.5400
4,KOSPI,2020-01-08,2151.3101
...,...,...,...
16625,SHANGHAI_COMP,2026-05-18,4131.5276
16626,SHANGHAI_COMP,2026-05-19,4169.5378
16627,SHANGHAI_COMP,2026-05-20,4162.1845
16628,SHANGHAI_COMP,2026-05-21,4077.2765
